# Rubicon xBD training on Kaggle (16 GB profile)

**Before running**
1. Upload the xBD archives as a Kaggle Dataset with this layout:
   ```
   /kaggle/input/<your-dataset>/xbd/
       train/{images,labels,targets}/
       hold/{images,labels,targets}/
   ```
2. Set **Accelerator** to `GPU T4 x2` or `GPU P100` (>= 15 GB VRAM).
3. Enable **Internet** so the notebook can clone the repo and fetch encoder weights.
4. Set the `XBD_DATASET_DIR` variable below if auto-discovery fails.

Training uses the event-disjoint split from `train/` (`mexico-earthquake` and
`palu-tsunami` held out). `hold/` is not used as a metric because it shares
event names with `train/`.

In [ ]:
import os, sys, json, shutil
from pathlib import Path

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(torch.cuda.get_device_name(0), '|', round(props.total_memory / 1024**3, 1), 'GB')
else:
    print('WARNING: no GPU. Training will fall back to the CPU profile and be very slow.')

In [ ]:
# Optional explicit override, e.g. Path('/kaggle/input/xbd-dataset/xbd')
XBD_DATASET_DIR = None

candidates = [
    Path(XBD_DATASET_DIR) if XBD_DATASET_DIR else None,
    Path('/kaggle/input/xbd'),
    Path('/kaggle/input/xbd-dataset/xbd'),
    Path('/kaggle/input/xview2/xbd'),
]
XBD_ROOT = next((p for p in candidates if p and (p / 'train').is_dir()), None)
if XBD_ROOT is None:
    for p in Path('/kaggle/input').rglob('train'):
        if (p / 'images').is_dir() and (p / 'labels').is_dir():
            XBD_ROOT = p.parent
            break
assert XBD_ROOT is not None, 'Could not find an xBD root with train/images and train/labels'
print('XBD_ROOT =', XBD_ROOT)
print('train pairs:', len(list((XBD_ROOT / 'train' / 'images').glob('*_pre_disaster.png'))))

In [ ]:
REPO = Path('/kaggle/working/Rubicon')
if not REPO.exists():
    !git clone -b Vanshika https://github.com/VikasKSingh05/Rubicon.git {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# Kaggle images already include torch/torchvision. Add the model + logging deps.
!pip install -q segmentation-models-pytorch==0.6.0 tensorboard
import segmentation_models_pytorch as smp
print('smp', smp.__version__)

In [ ]:
# Verify the real data and the event-disjoint split before spending GPU time.
import sys
sys.path.insert(0, str(REPO))
from ai_engine.data.check import run_data_check

stats = run_data_check(
    xbd_root_arg=str(XBD_ROOT),
    num_samples=2,
    crosscheck_n=3,
    samples_dir=Path('/kaggle/working/samples'),
)
assert stats['split']['val_events'], 'validation split is empty'
print('val events:', stats['split']['val_events'])

In [ ]:
OUT_DIR = Path('/kaggle/working/models')
LOG_DIR = Path('/kaggle/working/runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

!python train.py --config kaggle --epochs 30 \
  --xbd-root {XBD_ROOT} \
  --output-dir {OUT_DIR} \
  --log-dir {LOG_DIR}

In [ ]:
RESULTS_DIR = Path('/kaggle/working/results')
!python eval.py --checkpoint {OUT_DIR / 'checkpoint_best.pt'} \
  --xbd-root {XBD_ROOT} \
  --output-dir {RESULTS_DIR} \
  --batch-size 4

In [ ]:
metrics = json.loads((RESULTS_DIR / 'metrics.json').read_text())
print(json.dumps({
    'model_version': metrics['model_version'],
    'checkpoint_hash': metrics['checkpoint_hash'],
    'data_mode': metrics['data_mode'],
    'train_events': metrics['train_events'],
    'holdout_events': metrics['holdout_events'],
    'num_holdout_tiles': metrics['num_holdout_tiles'],
    'xview2_score': metrics['results']['xview2_score'],
    'loc_f1': metrics['results']['loc_f1'],
    'damage_f1_harmonic': metrics['results']['damage_f1_harmonic'],
}, indent=2))

## Collect artifacts

Download `checkpoint_best.pt`, `siamese_unet.pt`, `metrics.json`, and the
overlays from `/kaggle/working` (Output tab). Put the checkpoint in
`data/models/` locally and point `RUBICON_MODEL_DIR` at that directory for the
inference service.

Report metrics **per event** as well as the aggregate; two held-out events are
a coarse estimate and per-event variance is large.